In [1]:
import pandas as pd
import numpy as np
import statistics
import statsmodels.formula.api as smf
import scipy
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.diagnostic import (het_breuschpagan, het_white, acorr_breusch_godfrey,acorr_ljungbox)
from linearmodels.iv import IV2SLS

In [61]:
df = pd.read_excel("PS4data.xls",engine='xlrd')
data = df.copy()
print(data.describe())
data.head()

             year     quarter  real consumption of nondurables  \
count   220.00000  220.000000                       220.000000   
mean   1975.00000    2.500000                      1012.009545   
std      15.91071    1.120584                       414.399966   
min    1948.00000    1.000000                       442.300000   
25%    1961.00000    1.750000                       630.300000   
50%    1975.00000    2.500000                       941.950000   
75%    1989.00000    3.250000                      1341.200000   
max    2002.00000    4.000000                      1950.000000   

       real consumption of services         cpi  real disposable income  \
count                    220.000000  220.000000              220.000000   
mean                    1718.709091   77.034546             3390.639545   
std                      957.376853   52.588660             1724.012003   
min                      463.800000   23.533333             1055.300000   
25%                      836.8

,year,quarter,real consumption of nondurables,real consumption of services,cpi,real disposable income,population,equally weighted average return on the New York Stock Exchange
0,1948,1,442.3,463.8,23.533333,1055.3,102690.67,0.01414
1,1948,2,446.9,470.5,23.933333,1087.7,102915.33,0.11755
2,1948,3,443.3,476.6,24.466667,1107.1,103249.00,-0.09565
3,1948,4,449.6,480.2,24.233333,1109.8,103417.67,-0.04563
4,1949,1,451.8,482.6,23.866667,1087.8,103584.33,0.03338


In [41]:
data['consumption'] = data[['real consumption of nondurables','real consumption of services']].sum(axis=1)
data['per_capita_consumption'] = data[['consumption']].sum(axis = 1)/data[['population']].sum(axis=1)
print(data.nunique())
print(data['consumption'].describe())
data.head()

year                                                               55
quarter                                                             4
real consumption of nondurables                                   218
real consumption of services                                      219
cpi                                                               204
real disposable income                                            219
population                                                        220
equally weighted average return on the New York Stock Exchange    218
consumption                                                       220
per_capita_consumption                                            220
dtype: int64
count     220.000000
mean     2730.718636
std      1371.188225
min       906.100000
25%      1467.175000
50%      2483.350000
75%      3869.325000
max      5657.000000
Name: consumption, dtype: float64


,year,quarter,real consumption of nondurables,real consumption of services,cpi,real disposable income,population,equally weighted average return on the New York Stock Exchange,consumption,per_capita_consumption
0,1948,1,442.3,463.8,23.533333,1055.3,102690.67,0.01414,906.1,0.008824
1,1948,2,446.9,470.5,23.933333,1087.7,102915.33,0.11755,917.4,0.008914
2,1948,3,443.3,476.6,24.466667,1107.1,103249.00,-0.09565,919.9,0.008910
3,1948,4,449.6,480.2,24.233333,1109.8,103417.67,-0.04563,929.8,0.008991
4,1949,1,451.8,482.6,23.866667,1087.8,103584.33,0.03338,934.4,0.009021


In [42]:
lag_cols = []
for q in range(1,5):
    name = f"quarter_lag_{q}"
    data[name] = data['per_capita_consumption'].shift(q)
    lag_cols.append(name)

In [44]:
data = data.dropna()
data.head()

,year,quarter,real consumption of nondurables,real consumption of services,cpi,real disposable income,population,equally weighted average return on the New York Stock Exchange,consumption,per_capita_consumption,quarter_lag_1,quarter_lag_2,quarter_lag_3,quarter_lag_4
4,1949,1,451.8,482.6,23.866667,1087.8,103584.33,0.03338,934.4,0.009021,0.008991,0.008910,0.008914,0.008824
5,1949,2,452.7,485.7,23.866667,1091.3,103838.00,-0.08055,938.4,0.009037,0.009021,0.008991,0.008910,0.008914
6,1949,3,449.2,484.5,23.800000,1096.8,104127.33,0.13730,933.7,0.008967,0.009037,0.009021,0.008991,0.008910
7,1949,4,455.5,486.0,23.700000,1106.3,104427.67,0.12362,941.5,0.009016,0.008967,0.009037,0.009021,0.008991
8,1950,1,461.1,493.6,23.533333,1186.1,104733.33,0.05450,954.7,0.009116,0.009016,0.008967,0.009037,0.009021


In [58]:
x = sm.add_constant(data[lag_cols])
res = sm.OLS(data['per_capita_consumption'], x).fit()
print(het_breuschpagan(res.resid, x))               # heteroskedasticity
print(het_white(res.resid, x))                      # White's test (more general)
print(acorr_breusch_godfrey(res, nlags=4))         # serial correlation
print(acorr_ljungbox(res.resid, lags=[4, 8]))       # serial correlation

(np.float64(5.743106383457193), np.float64(0.21917222917134016), np.float64(1.4408510299779862), np.float64(0.22163788464858925))
(np.float64(9.754496643916433), np.float64(0.7799032746528425), np.float64(0.6790291159678654), np.float64(0.7933945015006704))
(np.float64(2.0209533437498832), np.float64(0.7319048002857307), 0.4887597041546031, 0.7439923264799301)
     lb_stat  lb_pvalue
4   0.307005   0.989358
8  13.555064   0.094125


In [59]:
res = sm.OLS(data['per_capita_consumption'], x).fit(cov_type='HC1')
print(res.summary())

                              OLS Regression Results                              
Dep. Variable:     per_capita_consumption   R-squared:                       1.000
Model:                                OLS   Adj. R-squared:                  1.000
Method:                     Least Squares   F-statistic:                 1.904e+05
Date:                    Wed, 11 Nov 2026   Prob (F-statistic):               0.00
Time:                            15:17:51   Log-Likelihood:                 1745.1
No. Observations:                     216   AIC:                            -3480.
Df Residuals:                         211   BIC:                            -3463.
Df Model:                               4                                         
Covariance Type:                      HC1                                         
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
const 

In [60]:
res.f_test("quarter_lag_2 = 0, quarter_lag_3 = 0, quarter_lag_4 = 0")

<class 'statsmodels.stats.contrast.ContrastResults'>
<F test: F=6.730700931605658, p=0.00023377665765220165, df_denom=211, df_num=3>

In [76]:
# 2SLS
data2SLS = df.copy()
data2SLS.head()

,year,quarter,real consumption of nondurables,real consumption of services,cpi,real disposable income,population,equally weighted average return on the New York Stock Exchange
0,1948,1,442.3,463.8,23.533333,1055.3,102690.67,0.01414
1,1948,2,446.9,470.5,23.933333,1087.7,102915.33,0.11755
2,1948,3,443.3,476.6,24.466667,1107.1,103249.00,-0.09565
3,1948,4,449.6,480.2,24.233333,1109.8,103417.67,-0.04563
4,1949,1,451.8,482.6,23.866667,1087.8,103584.33,0.03338


In [77]:
data2SLS['consumption'] = data2SLS[['real consumption of nondurables','real consumption of services']].sum(axis=1)
data2SLS['per_capita_consumption'] = data2SLS[['consumption']].sum(axis = 1)/data2SLS[['population']].sum(axis=1)
data2SLS.head()

,year,quarter,real consumption of nondurables,real consumption of services,cpi,real disposable income,population,equally weighted average return on the New York Stock Exchange,consumption,per_capita_consumption
0,1948,1,442.3,463.8,23.533333,1055.3,102690.67,0.01414,906.1,0.008824
1,1948,2,446.9,470.5,23.933333,1087.7,102915.33,0.11755,917.4,0.008914
2,1948,3,443.3,476.6,24.466667,1107.1,103249.00,-0.09565,919.9,0.008910
3,1948,4,449.6,480.2,24.233333,1109.8,103417.67,-0.04563,929.8,0.008991
4,1949,1,451.8,482.6,23.866667,1087.8,103584.33,0.03338,934.4,0.009021


In [78]:
data2SLS['income_per_capita'] = data2SLS[['real disposable income']].sum(axis = 1)/data2SLS[['population']].sum(axis=1)
data2SLS.head()

,year,quarter,real consumption of nondurables,real consumption of services,cpi,real disposable income,population,equally weighted average return on the New York Stock Exchange,consumption,per_capita_consumption,income_per_capita
0,1948,1,442.3,463.8,23.533333,1055.3,102690.67,0.01414,906.1,0.008824,0.010276
1,1948,2,446.9,470.5,23.933333,1087.7,102915.33,0.11755,917.4,0.008914,0.010569
2,1948,3,443.3,476.6,24.466667,1107.1,103249.00,-0.09565,919.9,0.008910,0.010723
3,1948,4,449.6,480.2,24.233333,1109.8,103417.67,-0.04563,929.8,0.008991,0.010731
4,1949,1,451.8,482.6,23.866667,1087.8,103584.33,0.03338,934.4,0.009021,0.010502


In [81]:
data2SLS['dc'] = np.log(data2SLS['per_capita_consumption']).diff()
data2SLS['dy'] = np.log(data2SLS['income_per_capita']).diff()
lag_cols = []
for q in range(2,6):
    name = f"dc_lag_{q}"
    data2SLS[name] = data2SLS['dc'].shift(q)
    lag_cols.append(name)

data2SLS.head(10)

,year,quarter,real consumption of nondurables,real consumption of services,cpi,real disposable income,population,equally weighted average return on the New York Stock Exchange,consumption,per_capita_consumption,income_per_capita,dc,dy,dc_lag_2,dc_lag_3,dc_lag_4,dc_lag_5
0,1948,1,442.3,463.8,23.533333,1055.3,102690.67,0.01414,906.1,0.008824,0.010276,NaN,NaN,NaN,NaN,NaN,NaN
1,1948,2,446.9,470.5,23.933333,1087.7,102915.33,0.11755,917.4,0.008914,0.010569,0.010209,0.028055,NaN,NaN,NaN,NaN
2,1948,3,443.3,476.6,24.466667,1107.1,103249.00,-0.09565,919.9,0.008910,0.010723,-0.000516,0.014442,NaN,NaN,NaN,NaN
3,1948,4,449.6,480.2,24.233333,1109.8,103417.67,-0.04563,929.8,0.008991,0.010731,0.009072,0.000804,0.010209,NaN,NaN,NaN
4,1949,1,451.8,482.6,23.866667,1087.8,103584.33,0.03338,934.4,0.009021,0.010502,0.003325,-0.021633,-0.000516,0.010209,NaN,NaN
5,1949,2,452.7,485.7,23.866667,1091.3,103838.00,-0.08055,938.4,0.009037,0.010510,0.001826,0.000766,0.009072,-0.000516,0.010209,NaN
6,1949,3,449.2,484.5,23.800000,1096.8,104127.33,0.13730,933.7,0.008967,0.010533,-0.007804,0.002245,0.003325,0.009072,-0.000516,0.010209
7,1949,4,455.5,486.0,23.700000,1106.3,104427.67,0.12362,941.5,0.009016,0.010594,0.005439,0.005744,0.001826,0.003325,0.009072,-0.000516
8,1950,1,461.1,493.6,23.533333,1186.1,104733.33,0.05450,954.7,0.009116,0.011325,0.011000,0.066727,-0.007804,0.001826,0.003325,0.009072
9,1950,2,466.1,507.6,23.700000,1178.1,105020.33,0.00104,973.7,0.009272,0.011218,0.016970,-0.009504,0.005439,-0.007804,0.001826,0.003325


In [82]:
data2SLS = data2SLS.dropna(subset = ['dc','dy']+lag_cols)
data2SLS.head()

,year,quarter,real consumption of nondurables,real consumption of services,cpi,real disposable income,population,equally weighted average return on the New York Stock Exchange,consumption,per_capita_consumption,income_per_capita,dc,dy,dc_lag_2,dc_lag_3,dc_lag_4,dc_lag_5
6,1949,3,449.2,484.5,23.800000,1096.8,104127.33,0.13730,933.7,0.008967,0.010533,-0.007804,0.002245,0.003325,0.009072,-0.000516,0.010209
7,1949,4,455.5,486.0,23.700000,1106.3,104427.67,0.12362,941.5,0.009016,0.010594,0.005439,0.005744,0.001826,0.003325,0.009072,-0.000516
8,1950,1,461.1,493.6,23.533333,1186.1,104733.33,0.05450,954.7,0.009116,0.011325,0.011000,0.066727,-0.007804,0.001826,0.003325,0.009072
9,1950,2,466.1,507.6,23.700000,1178.1,105020.33,0.00104,973.7,0.009272,0.011218,0.016970,-0.009504,0.005439,-0.007804,0.001826,0.003325
10,1950,3,474.4,514.6,24.266667,1196.5,105248.33,0.16630,989.0,0.009397,0.011368,0.013422,0.013329,0.011000,0.005439,-0.007804,0.001826


In [85]:
#2SLS IV regression lags are the instrument variables

res = IV2SLS(dependent=data2SLS['dc'],
             exog = sm.add_constant(data2SLS[[]]),
             endog = data2SLS['dy'],
             instruments = data2SLS[lag_cols],
             ).fit(cov_type='robust')
print(res.summary)

                          IV-2SLS Estimation Summary                          
Dep. Variable:                     dc   R-squared:                      0.0685
Estimator:                    IV-2SLS   Adj. R-squared:                 0.0641
No. Observations:                 214   F-statistic:                    4.6529
Date:                Thu, Nov 12 2026   P-value (F-stat)                0.0310
Time:                        11:46:31   Distribution:                  chi2(1)
Cov. Estimator:                robust                                         
                                                                              
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
const          0.0026     0.0012     2.2579     0.0240      0.0003      0.0049
dy             0.4369     0.2026     2.1571     0.03

In [90]:
#python tests
print(res.first_stage)
print(res.wu_hausman())
print(res.sargan)
print(res.anderson_rubin)

    First Stage Estimation Results    
                                    dy
--------------------------------------
R-squared                       0.0458
Partial R-squared               0.0458
Shea's R-squared                0.0458
Partial F-statistic             12.832
P-value (Partial F-stat)        0.0121
Partial F-stat Distn           chi2(4)
========================== ===========
const                           0.0040
                              (2.4066)
dc_lag_2                       -0.0186
                             (-0.0960)
dc_lag_3                        0.3372
                              (2.9366)
dc_lag_4                        0.1704
                              (1.4466)
dc_lag_5                       -0.2219
                             (-1.3821)
--------------------------------------

T-stats reported in parentheses
T-stats use same covariance type as original model
Wu-Hausman test of exogeneity
H0: All endogenous variables are exogenous
Statistic: 1.8676
P-valu

In [100]:
W = sm.add_constant(data2SLS[lag_cols])            # constant + the 4 instruments

# 1. Wu-Hausman (control function): first-stage residual added to structural eq.
fs = sm.OLS(data2SLS["dy"], W).fit()
data2SLS["vhat"] = fs.resid
cf = sm.OLS(data2SLS["dc"], sm.add_constant(data2SLS[["dy", "vhat"]])).fit()
print("Wu-Hausman Test Results:",np.round(cf.tvalues["vhat"]**2,5), np.round(cf.pvalues["vhat"],5))   # F = t^2 for one restriction
# cross-check: res.wu_hausman()  (res = your IV2SLS fit)

# 2. Sargan, straight from the formula in the notes
e = res.resids.values
Wm = W.values
S = len(e) * (e @ Wm @ np.linalg.inv(Wm.T @ Wm) @ Wm.T @ e) / (e @ e)
print("Sargan Test:",np.round(S,5), np.round(1 - stats.chi2.cdf(S, 3),5))                 # chi2(l-k) = chi2(3)
# cross-check: res.sargan

# 3. Stock-Yogo: classical first-stage F on the 4 instruments
print("Stock-Yogo:",fs.f_test(" = 0, ".join(lag_cols) + " = 0"))

# 4. Anderson-Rubin confidence set for gamma1
keep = []
for g in np.linspace(-1, 2, 601):
    r = sm.OLS(data2SLS["dc"] - g * data2SLS["dy"], W).fit()
    f = r.f_test(" = 0, ".join(lag_cols) + " = 0")
    if float(f.pvalue) > 0.05:
        keep.append(g)
print("Anderson-Rubin Confidence Set:",np.round(min(keep),4), max(keep))

Wu-Hausman Test Results: 1.8676 0.1732
Sargan Test: 7.73828 0.05174
Stock-Yogo: <F test: F=2.50911522090202, p=0.04303997151918791, df_denom=209, df_num=4>
Anderson-Rubin Confidence Set: 0.325 2.0


In [3]:
stock = pd.read_excel('SP500Index.xlsx')
stock.head()

,Date of Observation,Level of the S&P 500 Index
0,19600129,55.61
1,19600229,56.12
2,19600331,55.34
3,19600429,54.37
4,19600531,55.83


In [5]:
SP_0 = stock['Level of the S&P 500 Index'].iloc[0]
SP_0

np.float64(55.61)

In [6]:
stock['SPt / SP0'] = stock['Level of the S&P 500 Index'] / SP_0
stock.head()

,Date of Observation,Level of the S&P 500 Index,SPt / SP0
0,19600129,55.61,1.000000
1,19600229,56.12,1.009171
2,19600331,55.34,0.995145
3,19600429,54.37,0.977702
4,19600531,55.83,1.003956


In [9]:
stock['log(SPt/SP)'] = np.log(stock['SPt / SP0'])
stock.head(10)

,Date of Observation,Level of the S&P 500 Index,SPt / SP0,log(SPt/SP)
0,19600129,55.61,1.000000,0.000000
1,19600229,56.12,1.009171,0.009129
2,19600331,55.34,0.995145,-0.004867
3,19600429,54.37,0.977702,-0.022551
4,19600531,55.83,1.003956,0.003948
5,19600630,56.92,1.023557,0.023284
6,19600729,55.51,0.998202,-0.001800
7,19600831,56.96,1.024276,0.023986
8,19600930,53.52,0.962417,-0.038308
9,19601031,53.39,0.960079,-0.040740


In [10]:
stock.describe()

,Date of Observation,Level of the S&P 500 Index,SPt / SP0,log(SPt/SP)
count,7.020000e+02,702.000000,702.000000,702.000000
mean,1.988820e+07,642.002208,11.544726,1.783789
std,1.690036e+05,668.416584,12.019719,1.219625
min,1.960013e+07,53.390000,0.960079,-0.040740
25%,1.974086e+07,99.205000,1.783942,0.578823
50%,1.989038e+07,313.810000,5.643050,1.730336
75%,2.003110e+07,1146.777500,20.621786,3.026348
max,2.018063e+07,2823.810000,50.778817,3.927479


In [11]:
np.random.seed(162)

In [25]:
xt = stock['log(SPt/SP)']
delta_mle = xt.diff().mean()
delta_mle

np.float64(0.005548394684384916)

In [29]:
T = len(xt) - 1
sigma2_mle = xt.diff().var(ddof = 0)
sigma2_mle

np.float64(0.0017791815949153168)

In [39]:
sigma = np.sqrt(sigma2_mle)
sigma

np.float64(0.04218034607391595)

In [40]:
#check
simga2_mle2 = xt.diff().std(ddof = 0)
simga2_mle2

np.float64(0.04218034607391595)

In [38]:
stock['xt'] = stock['log(SPt/SP)'].diff()
delta_ml2 = stock['xt'].mean()
delta_ml2

np.float64(0.005548394684384916)

In [37]:
#check
delta_ml3 = stock['log(SPt/SP)'].iloc[-1] / T
delta_ml3

np.float64(0.005548394684384916)

In [41]:
inter_delta_mle = np.exp(delta_mle)
inter_delta_mle

np.float64(1.0055638155332989)